START OF CODE

# START OF CODE - IMPORT AND INIT
---

In [ ]:
# INSTALL IN ACTIVE NOTEBOOK KERNEL
%pip install python-dotenv google-genai openai anthropic matplotlib

In [ ]:
# Core utilities
from dotenv import load_dotenv
import os
import time
import re
import json

# Data + modeling
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from scipy.spatial.distance import jensenshannon
from collections import defaultdict
import matplotlib.pyplot as plt

# Gemini API SDK
import google.genai as genai
from google.genai import types

# ChatGPT API SDK
from openai import OpenAI

# Claude API SDK
from anthropic import Anthropic

print("Imports ready:", pd.__version__)

In [ ]:
# Load environment variables once.
load_dotenv()

# API keys (kept in one place for easier debugging)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

# Client registry: initialize available clients and keep missing ones as None.
clients = {
    'gemini': genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None,
    'chatgpt': OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None,
    'claude': Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None,
}

print("Client status:")
for name, c in clients.items():
    print(f"- {name}: {'ready' if c is not None else 'missing API key'}")

# Start of Persona 0
---

## AI Client test

In [ ]:
# AI test: Generate a simple response (optional - may encounter rate limits)

# try:
#     response = client.models.generate_content(
#         model="gemini-2.5-flash", contents="Give me a quick and simple reply."
#     )
#     print(response.text)
# except Exception as e:
#     print(f"API test skipped (may be temporary): {type(e).__name__}")

## Data Processing

In [ ]:
# 1) Load survey data
df = pd.read_csv('jgss_dummy_data.csv')

# 2) Define variable groups used in feature selection and evaluation
core_demographics = [
    'Sex', 'Age', 'Region', 'Area', 'Education', 
    'Income', 'Occupation', 'Marital_Status'
]

# Held-out outcomes for persona evaluation (not used in feature ranking)
outcome_variables = [
    'ep01_gen_economic_sit', 'ep03_own_economic_sit', 'ep06_future_economic_sit',
    'mp12_foreigners_help_shortage', 'mn01_born_in_country_importance', 
    'mp18_refugees_opportunities_risks', 'li03_leisure_importance', 
    'lp03_ordinary_people_worse', 'lp04_unjustifiable_children', 'ca12_smoking_hashish', 
    'vm16_own_cells_ivf', 'vm17_donated_cells_ivf', 'pt15_trust_political_parties', 
    'pe05_politicians_represent_interests', 'ps01_satisfaction_government', 
    'rb01_personal_god', 'mm01_restrict_islam', 'mm06_religious_fanatics', 
    'st01_trust_people', 'sm01_union_member_current', 'sm02_union_member_past', 
    'im03_importance_education', 'im08_importance_diligence', 'iw04_state_welfare', 
    'vi06_help_disadvantaged', 'vi07_assert_needs', 'vi10_politically_active'
]

# Candidate predictors: everything except IDs, demographics, and held-out outcomes
features_to_rank = [
    col for col in df.columns
    if col not in core_demographics and col not in outcome_variables and col != 'Respondent_ID'
]

# 3) Impute missing values so each RF model can train without split-ballot null errors
imputer = SimpleImputer(strategy='most_frequent')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

In [ ]:
# 4) For each candidate variable, train an RF classifier and accumulate top predictors
k = 5  # Top predictors kept per target model
master_importance_scores = defaultdict(float)

for target_var in features_to_rank:
    # Predict one variable from the rest (leave-one-target-out)
    X = df_imputed[features_to_rank].drop(columns=[target_var])
    y = df_imputed[target_var].astype(str)  # Force classification mode

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X, y)

    importances = pd.Series(rf.feature_importances_, index=X.columns)
    top_k = importances.nlargest(k)

    # Aggregate importance across all target models to get globally useful variables
    for feature, score in top_k.items():
        master_importance_scores[feature] += score

In [ ]:
# 5. Sort the master dictionary to find your global TOP-k variables
sorted_ranking = sorted(master_importance_scores.items(), key=lambda x: x[1], reverse=True)

# 6. Display the Results
print("\n--- MASTER VARIABLE IMPORTANCE RANKING ---")
for i, (var, score) in enumerate(sorted_ranking[:k]): # Displaying TOP-k
    print(f"Rank {i+1}: {var} (Aggregated Score: {score:.4f})")

## Persona Creation

In [ ]:
# Build compact JSON persona profiles from demographics + top-k ranked attributes
personas = []

print(sorted_ranking[:k])
top_k_variables = [var for var, _ in sorted_ranking[:k]]
print(top_k_variables)

for index, row in df.iterrows():
    persona_dict = {}

    # Keep only non-missing demographic fields
    for demo in core_demographics:
        if pd.notna(row[demo]):
            persona_dict[demo] = row[demo]

    # Add non-missing high-signal attributes discovered by RF ranking
    for top_var in top_k_variables:
        if pd.notna(row[top_var]):
            persona_dict[top_var] = row[top_var]

    # Store as JSON string so it can be passed directly to prompt text
    persona_json = json.dumps(persona_dict, ensure_ascii=False)
    personas.append(persona_json)

print(f"Successfully generated {len(personas)} JSON personas.")
print("\n--- EXAMPLE PERSONA (Ready for API) ---")
print(personas[0])

## Persona Prompting -- Multiple Questions

Question bank holds the parameters for prompting and evaluation

In [ ]:
# Question config: prompt text + valid response scale for each target variable
question_bank = {
    'ep01_gen_economic_sit': {
        'question': "現在の一般的な経済状況についてどう思いますか？ 1 (非常に良い) から 5 (非常に悪い) でお答えください。数字のみを出力してください。",
        'scale_min': 1,
        'scale_max': 5,
    },
    'pt15_trust_political_parties': {
        'question': "政党をどの程度信頼していますか？ 1 (全く信頼していない) から 5 (非常に信頼している) でお答えください。数字のみを出力してください。",
        'scale_min': 1,
        'scale_max': 5,
    },
    'st01_trust_people': {
        'question': "一般的に、人は信頼できると思いますか？ 1 (信頼できる) から 4 (信頼できない) でお答えください。数字のみを出力してください。",
        'scale_min': 1,
        'scale_max': 4,
    }
}

# Central runtime config so parameters are not scattered across cells.
MODEL_CONFIG = {
    'gemini': {'model_id': 'gemini-2.5-flash', 'sleep_seconds': 1.0},
    'chatgpt': {'model_id': 'gpt-5.4-mini', 'sleep_seconds': 1.0},
    'claude': {'model_id': 'claude-3-5-sonnet-latest', 'sleep_seconds': 1.0},
}

sys_instruct = "あなたは日本のアンケート回答者です。提供されたペルソナプロファイルに基づいて、質問に答えてください。出力は1桁の数字のみにしてください。"

print(f"Starting generalized simulation for {len(question_bank)} questions and {len(personas)} personas...")

# Keep generated answers separate from the original survey dataframe.
results_df = pd.DataFrame(index=df.index)
if 'Respondent_ID' in df.columns:
    results_df['Respondent_ID'] = df['Respondent_ID']

# Collect JSD and run diagnostics for all models in one table.
metrics_rows = []
metrics_df = pd.DataFrame()

# Precompute real distributions once and reuse across models.
real_distributions = {
    var_name: df[var_name].value_counts(normalize=True).sort_index().reindex(
        range(spec['scale_min'], spec['scale_max'] + 1),
        fill_value=0,
    )
    for var_name, spec in question_bank.items()
}


def build_user_prompt(persona_json, question_text):
    return f"Persona Profile:\n{persona_json}\n\nQuestion:\n{question_text}"


def extract_first_int(raw_text):
    match = re.search(r"\d+", raw_text or "")
    return int(match.group()) if match else None


def reset_model_outputs(model_label):
    global metrics_rows
    model_cols = [c for c in results_df.columns if c.startswith(f"{model_label}__")]
    if model_cols:
        results_df.drop(columns=model_cols, inplace=True)
    metrics_rows = [r for r in metrics_rows if r.get('model') != model_label]


def record_metrics(model_label, var_name, simulated_answers, all_values):
    real_dist = real_distributions[var_name]
    sim_series = pd.Series(simulated_answers)
    sim_dist = sim_series.value_counts(normalize=True).sort_index().reindex(all_values, fill_value=0)
    valid_n = int(sim_series.notna().sum())

    print(f"Real distribution for {var_name}:", real_dist.values)
    print(f"Simulated distribution for {var_name}:", sim_dist.values)

    if sim_dist.sum() == 0:
        print(f"JSD Score for {var_name}: skipped (no valid simulated responses)")
        jsd_score = None
    else:
        jsd_score = float(jensenshannon(real_dist, sim_dist))
        print(f"JSD Score for {var_name}: {jsd_score:.4f}")

    return {
        'model': model_label,
        'question_var': var_name,
        'answer_column': f"{model_label}__{var_name}",
        'valid_answers': valid_n,
        'total_personas': len(personas),
        'jsd': jsd_score,
    }


def run_model_simulation(model_label, generate_raw_text_fn):
    global metrics_rows, metrics_df

    reset_model_outputs(model_label)
    sleep_seconds = MODEL_CONFIG[model_label]['sleep_seconds']

    for var_name, spec in question_bank.items():
        question_text = spec['question']
        all_values = range(spec['scale_min'], spec['scale_max'] + 1)
        simulated_answers = []

        print(f"\n--- [{model_label}] Testing Variable: {var_name} ---")

        for i, persona_json in enumerate(personas):
            user_prompt = build_user_prompt(persona_json, question_text)
            try:
                raw_text = generate_raw_text_fn(user_prompt)
                answer = extract_first_int(raw_text)
                simulated_answers.append(answer)
                print(f"Respondent {i+1} replied: {answer}")
                time.sleep(sleep_seconds)
            except Exception as e:
                print(f"API Error on respondent {i+1}: {e}")
                simulated_answers.append(None)

        answer_col = f"{model_label}__{var_name}"
        results_df[answer_col] = simulated_answers
        metrics_rows.append(record_metrics(model_label, var_name, simulated_answers, all_values))

    metrics_df = pd.DataFrame(metrics_rows)
    print(f"\n{model_label} run complete. results_df columns:", [c for c in results_df.columns if c.startswith(f"{model_label}__")])


print("Shared simulation configuration and helper functions are ready.")

## Gemini

In [ ]:
def gemini_generate_raw_text(user_prompt):
    gemini_client = clients['gemini']
    if gemini_client is None:
        raise ValueError("GEMINI_API_KEY is missing")

    response = gemini_client.models.generate_content(
        model=MODEL_CONFIG['gemini']['model_id'],
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            temperature=0.0,
        ),
    )
    return response.text or ""


run_model_simulation('gemini', gemini_generate_raw_text)

## ChatGPT

In [ ]:
def chatgpt_generate_raw_text(user_prompt):
    chatgpt_client = clients['chatgpt']
    if chatgpt_client is None:
        raise ValueError("OPENAI_API_KEY is missing")

    response = chatgpt_client.responses.create(
        model=MODEL_CONFIG['chatgpt']['model_id'],
        instructions=sys_instruct,
        input=user_prompt,
        temperature=0.0,
    )
    return response.output_text or ""


run_model_simulation('chatgpt', chatgpt_generate_raw_text)

## Claude

In [ ]:
def claude_generate_raw_text(user_prompt):
    claude_client = clients['claude']
    if claude_client is None:
        raise ValueError("ANTHROPIC_API_KEY is missing")

    response = claude_client.messages.create(
        model=MODEL_CONFIG['claude']['model_id'],
        system=sys_instruct,
        max_tokens=10,
        temperature=0.0,
        messages=[
            {"role": "user", "content": user_prompt}
        ],
    )
    return "".join(block.text for block in response.content if hasattr(block, "text"))


run_model_simulation('claude', claude_generate_raw_text)

## Results Visualization

Run this section after the Gemini, ChatGPT, and Claude cells to compare model quality and response behavior.

In [ ]:

# Basic guards so this cell fails gracefully if run too early.
if 'metrics_df' not in globals() or metrics_df.empty:
    raise ValueError("metrics_df is empty. Run the three model cells first.")
if 'results_df' not in globals() or results_df.empty:
    raise ValueError("results_df is empty. Run the three model cells first.")

model_order = [m for m in ['gemini', 'chatgpt', 'claude'] if m in metrics_df['model'].unique()]
if not model_order:
    raise ValueError("No model results found in metrics_df.")

metrics_plot_df = metrics_df.copy()
metrics_plot_df['jsd'] = pd.to_numeric(metrics_plot_df['jsd'], errors='coerce')
metrics_plot_df['valid_rate'] = metrics_plot_df['valid_answers'] / metrics_plot_df['total_personas']

# 1) JSD comparison by question and model (lower is better).
jsd_pivot = metrics_plot_df.pivot(index='question_var', columns='model', values='jsd').reindex(columns=model_order)
print("JSD comparison table (lower is better):")
print(jsd_pivot.round(4))

ax = jsd_pivot.plot(kind='bar', figsize=(10, 4), rot=20)
ax.set_title('JSD by Question and Model (Lower is Better)')
ax.set_xlabel('Question Variable')
ax.set_ylabel('JSD')
ax.legend(title='Model')
plt.tight_layout()
plt.show()

# 2) Valid answer-rate comparison.
valid_pivot = metrics_plot_df.pivot(index='question_var', columns='model', values='valid_rate').reindex(columns=model_order)
print("\nValid answer-rate table:")
print(valid_pivot.round(3))

ax = valid_pivot.plot(kind='bar', figsize=(10, 4), rot=20)
ax.set_title('Valid Answer Rate by Question and Model')
ax.set_xlabel('Question Variable')
ax.set_ylabel('Valid Answer Rate')
ax.set_ylim(0, 1.05)
ax.legend(title='Model')
plt.tight_layout()
plt.show()

# 3) Response distribution for one selected question.
viz_var = 'ep01_gen_economic_sit'  # Change this to another key in question_bank as needed.
if viz_var not in question_bank:
    raise ValueError(f"{viz_var} not found in question_bank")

all_values = list(range(question_bank[viz_var]['scale_min'], question_bank[viz_var]['scale_max'] + 1))
dist_data = {
    'real': df[viz_var].value_counts(normalize=True).sort_index().reindex(all_values, fill_value=0)
}

for model_name in model_order:
    col_name = f"{model_name}__{viz_var}"
    if col_name in results_df.columns:
        dist_data[model_name] = (
            results_df[col_name].value_counts(normalize=True).sort_index().reindex(all_values, fill_value=0)
        )

dist_df = pd.DataFrame(dist_data, index=all_values)
print(f"\nDistribution comparison for {viz_var}:")
print(dist_df.round(3))

ax = dist_df.plot(kind='bar', figsize=(10, 4), rot=0)
ax.set_title(f'Real vs Simulated Distribution ({viz_var})')
ax.set_xlabel('Answer Value')
ax.set_ylabel('Probability')
ax.legend(title='Source')
plt.tight_layout()
plt.show()

---
---

## Persona Prompting -- Single Question (Outdated)

Original code, do not use

In [ ]:
# import re
# import time
# from google.genai import types

# # Corrected API call block
# persona_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
# model_id = "gemini-2.5-flash"

# question_text = "現在の一般的な経済状況についてどう思いますか？ 1 (非常に良い) から 5 (非常に悪い) でお答えください。数字のみを出力してください。"
# sys_instruct = "あなたは日本のアンケート回答者です。提供されたペルソナプロファイルに基づいて、質問に答えてください。出力は1桁の数字のみにしてください。"

# simulated_answers = []
# print(f"Starting LLM Simulation for {len(personas)} personas...")

# for i, persona_json in enumerate(personas):
#     user_prompt = f"Persona Profile:\n{persona_json}\n\nQuestion:\n{question_text}"
#     try:
#         response = persona_client.models.generate_content(
#             model=model_id,
#             contents=user_prompt,
#             config=types.GenerateContentConfig(
#                 system_instruction=sys_instruct,
#                 temperature=0.0,
#             ),
#         )
#         raw_text = response.text or ""
#         match = re.search(r"\d+", raw_text)
#         answer = int(match.group()) if match else None
#         simulated_answers.append(answer)
#         print(f"Respondent {i+1} replied: {answer}")
#         time.sleep(1)
#     except Exception as e:
#         print(f"API Error on respondent {i+1}: {e}")
#         simulated_answers.append(None)

# df["simulated_ep01"] = simulated_answers
# print("\nSimulation Complete!")

### Evaluation

In [ ]:
# # Count the frequencies of each answer (1 through 5) and normalize to probabilities
# real_dist = df['ep01_gen_economic_sit'].value_counts(normalize=True).sort_index()
# sim_dist = df['simulated_ep01'].value_counts(normalize=True).sort_index()

# # Ensure both distributions cover all possible values (1-5), filling missing values with 0
# all_values = range(1, 6)
# real_dist = real_dist.reindex(all_values, fill_value=0)
# sim_dist = sim_dist.reindex(all_values, fill_value=0)

# print("Real distribution:", real_dist.values)
# print("Simulated distribution:", sim_dist.values)

# # Calculate the JSD
# jsd_score = jensenshannon(real_dist, sim_dist)
# print(f"\nJSD Score: {jsd_score:.4f}")
# print("(Lower scores indicate distributions are more similar)")